<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T06-L04-agent-memory/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T06-L04-agent-memory/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T06-L04-agent-memory/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T06-L04-agent-memory/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T06-L04 · Agent memory: what to keep, what to retrieve, what to forget

**You will build:** a memory store with a write policy and a recency+importance+relevance
retrieval function, measured by FACT RECALL against memory size and retrieval cost.
**Time:** ~45 minutes · **Runs on:** a laptop CPU, 8 GiB RAM ·
**Prerequisites:** T00-L01-the-8gb-track, T06-L01-retrieval-from-scratch,
T07-L01-agent-loop-from-scratch.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T00-L01`, `T06-L01`, `T07-L01`

**This lesson assumes you can already:**

- keep a store of records in Python (a list of dicts or a small class) and update or filter them in a loop
- compute the cosine similarity of two vectors and say why they are length-normalised first
- explain exponential decay (value times a constant to the power of elapsed time) and which direction the constant moves the curve
- rescale a list of numbers to the range 0 to 1 (min-max scaling), and say what happens when they are all equal

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Generative Agents: Interactive Simulacra of Human Behavior](https://arxiv.org/abs/2304.03442) · Park, O'Brien, Cai, Morris, Liang and Bernstein (arXiv; UIST 2023) · *free* — section 4.1, Memory and Retrieval: the recency, importance and relevance score, with exponential-decay recency and min-max scaling, that exercises 5 and 6 implement
- [LLM Powered Autonomous Agents (the Memory section)](https://lilianweng.github.io/posts/2023-06-23-agent/) · Lilian Weng (Lil'Log) · *free* — a readable map of short-term versus long-term memory in agents and why a retriever sits between the store and the prompt, before you build one
- [Mem0: Building Production-Ready AI Agents with Scalable Long-Term Memory](https://arxiv.org/abs/2504.19413) · Chhikara, Khant, Aryan, Singh and Yadav (arXiv) · *free* — its update phase picks ADD, UPDATE, DELETE or NOOP for each new fact against the most similar stored memories, an LLM-driven version of the dedup-and-supersede rules in exercises 2 to 4
- [Introduction to Information Retrieval, section on dot products and cosine similarity (chapter 6)](https://nlp.stanford.edu/IR-book/html/htmledition/dot-products-1.html) · Manning, Raghavan and Schütze (Cambridge University Press; free online edition) · *free* — cosine similarity and length normalisation worked through on TF-IDF vectors, which is the relevance signal in the retrieval score

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import re
import sys
import traceback
from typing import Callable, NamedTuple, Optional, Sequence

import numpy as np

print("python", sys.version.split()[0], "· numpy", np.__version__, "· platform", sys.platform)

SEED = 20261014          # this lesson's one fixed seed: the interaction log, every value the
                          # user's facts take, and every probe question all derive from it, so
                          # re-running this file prints the same numbers on any machine.
N_TURNS = 480             # length of the synthetic interaction log, in turns.
PROBE_EVERY = 40          # ask about every known slot's CURRENT value this often, after warm-up.
THRESHOLD = 2.5           # the salience gate exercise 1 and the demo below use by default.
SIM_THRESHOLD = 0.35      # the near-duplicate cosine gate exercise 2 and the demo use by default.
DECAY = 0.99              # Park et al. (2023)'s own value is 0.995 per game HOUR; this lesson's
                          # unit of time is one interaction TURN, and a faster fade per turn is
                          # a teaching choice, not a value taken from the paper.

_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("write_policy_threshold",),
    "exercise 2": ("find_duplicate",),
    "exercise 3": ("merge_or_append",),
    "exercise 4": ("apply_update",),
    "exercise 5": ("recency_score",),
    "exercise 6": ("retrieve_top_k",),
}


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (merge_or_append)"; several -> "exercises 3, 5 and 6"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. The problem: a long-running assistant cannot keep everything in context

A conversation that keeps going eventually outgrows any fixed context window. A
long-running agent's answer is to write some of what it hears to a MEMORY STORE — outside
the context window — and later RETRIEVE a handful of the most useful memories back into the
prompt. Two decisions govern how well that works, and both are graded below:

- **What to keep.** Store everything, and the store fills with restatements and small talk
  that later crowd out anything else a small retrieval budget could have held. Store nothing
  but the highest-salience facts, and you keep the budget clean but risk missing something
  asked about later. And when the user CORRECTS a fact — moves city, changes team — the old
  value has to stop being the answer, not just start losing to the new one on points.
- **What to retrieve.** Given a question, which few memories out of possibly hundreds are
  worth spending the budget on? Park et al. (2023)'s *Generative Agents* answers this with
  three signals — how RECENT a memory is, how IMPORTANT it seemed, how RELEVANT it is to the
  question — combined into one score (`claims.yaml`, `retrieval-score-combination`).

This lesson builds a small memory system exactly that shape, over a synthetic conversation
log where facts get introduced, restated in different words, and later changed — and
measures FACT RECALL: for a question asked long after the fact was last mentioned, does the
system return the memory holding the CURRENT value?

## 2. Given: the tokeniser and the embedding it feeds

`tokenize` is carried forward verbatim from T06-L01 — the same regex, the same behaviour.
The embedding here is plainer than T06-L01's: a single L2-normalised TF-IDF vector with no
SVD step, because RELEVANCE in Park et al. (2023) is "the cosine similarity between the
memory's embedding vector and the query memory's embedding vector" — a single cosine, not a
ranked retrieval task — so nothing here needs T06-L01's dimensionality reduction.

In [ ]:
_TOKEN_RE = re.compile(r"[a-z0-9]+(?:-[a-z0-9]+)*")   # carried forward from T06-L01, verbatim


def tokenize(text: str) -> list[str]:
    """Lower-case `text` and split it into tokens (carried forward from T06-L01, verbatim).

    Example:
        >>> tokenize("I'm based in Lisbon now.")
        ['i', 'm', 'based', 'in', 'lisbon', 'now']
    """
    return _TOKEN_RE.findall(text.lower())


def build_vocab(texts: Sequence[str]) -> list[str]:
    """Given: every distinct token across `texts`, sorted — this lesson's fixed vocabulary."""
    vocab: set[str] = set()
    for text in texts:
        vocab.update(tokenize(text))
    return sorted(vocab)


def build_idf(texts: Sequence[str], vocab: Sequence[str]) -> np.ndarray:
    """Given: smoothed inverse document frequency over `vocab`, one weight per word.

    `idf[w] = log((n + 1) / (df[w] + 1)) + 1` — a smoothed cousin of T06-L01's `log(N / n_t) + 1`.
    The extra +1s matter here: the vocabulary also holds the probe questions' words, which
    appear in no logged text (`df = 0`), and T06-L01's form would divide by zero on them.
    """
    index = {w: i for i, w in enumerate(vocab)}
    df = np.zeros(len(vocab), dtype=float)
    for text in texts:
        for w in set(tokenize(text)):
            j = index.get(w)
            if j is not None:
                df[j] += 1.0
    n = len(texts)
    return np.log((n + 1) / (df + 1.0)) + 1.0


def embed(text: str, vocab: Sequence[str], idf: np.ndarray) -> np.ndarray:
    """Given: `text` as an L2-normalised TF-IDF vector over `vocab`.

    A word outside `vocab` is ignored. A vector of all zeros (no known word matched) is
    returned as-is, not divided by a zero norm.
    """
    index = {w: i for i, w in enumerate(vocab)}
    vec = np.zeros(len(vocab), dtype=float)
    for tok in tokenize(text):
        j = index.get(tok)
        if j is not None:
            vec[j] += 1.0
    vec *= idf
    norm = float(np.linalg.norm(vec))
    return vec / norm if norm > 0.0 else vec


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    """Given: cosine similarity between `a` and `b`; 0.0 if either is the zero vector."""
    na, nb = float(np.linalg.norm(a)), float(np.linalg.norm(b))
    if na == 0.0 or nb == 0.0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))


def _check_embedding_toolkit() -> None:
    v = build_vocab(["pump note", "note valve"])
    assert v == ["note", "pump", "valve"], f"build_vocab should sort the distinct tokens, got {v}"
    idf = build_idf(["pump note", "note valve"], v)
    a, b = embed("pump note", v, idf), embed("note valve", v, idf)
    assert abs(np.linalg.norm(a) - 1.0) < 1e-9, "embed() should L2-normalise a non-empty vector"
    assert 0.0 < cosine(a, b) < 1.0, "two texts sharing one word out of two should cosine strictly between 0 and 1"
    print("embedding toolkit looks right")


_try("embedding toolkit (given)", _check_embedding_toolkit)

## 3. Given: the interaction log

Six facts about a fictional user, each with its own base importance and several paraphrase
templates. Over `N_TURNS` turns the user states each fact for the first time, later restates
it in different words, sometimes CHANGES it (moves city, gets a new manager), and in between,
says things with nothing to remember. Read the generator — knowing exactly how the log is
built is what lets you predict, before running anything, which policy should win.

In [ ]:
SLOTS: dict[str, dict] = {
    "home_city": {
        "base_importance": 8.5,
        "values": ["Utrecht", "Rotterdam", "Groningen", "Berlin", "Lisbon", "Austin"],
        "templates": [
            "I live in {v} now.",
            "My home city these days is {v}.",
            "I'm based in {v} at the moment.",
            "Home for me right now is {v}.",
        ],
    },
    "role": {
        "base_importance": 7.0,
        "values": ["research engineer", "platform lead", "data scientist", "product manager"],
        "templates": [
            "I work as a {v}.",
            "My job title is {v}.",
            "These days I'm a {v}.",
            "I'm employed as a {v}.",
        ],
    },
    "manager": {
        "base_importance": 5.5,
        "values": ["Priya", "Daan", "Noor", "Felix"],
        "templates": [
            "My manager is {v}.",
            "{v} is who I report to.",
            "I report to {v} these days.",
            "{v} manages me right now.",
        ],
    },
    "team_size": {
        "base_importance": 4.5,
        "values": ["3", "6", "11", "20"],
        "templates": [
            "My team has {v} people.",
            "There are {v} of us on the team.",
            "Team size right now is {v}.",
            "I work with {v} teammates.",
        ],
    },
    "commute": {
        "base_importance": 3.5,
        "values": ["bike", "train", "car", "walk"],
        "templates": [
            "I commute by {v}.",
            "I get to work by {v}.",
            "These days I {v} to the office.",
            "My commute is by {v}.",
        ],
    },
    "coffee_order": {
        "base_importance": 3.0,
        "values": ["flat white", "oat latte", "espresso", "americano"],
        "templates": [
            "My usual order is a {v}.",
            "I always get a {v}.",
            "These days I drink {v}.",
            "My go-to coffee is a {v}.",
        ],
    },
}

NOISE_PHRASES = [
    "Nice weather today.",
    "Traffic was fine this morning.",
    "Looking forward to the weekend.",
    "That meeting ran long.",
    "The office coffee machine is broken again.",
    "Quick lunch today, back in five.",
    "Almost forgot my badge this morning.",
    "Long week already.",
]

REPEAT_DISCOUNT = 0.6     # a restatement of an already-known fact is less novel than learning it


def importance_oracle(slot: Optional[str], kind: str) -> float:
    """Given: a stand-in for "ask the language model how poignant this is" (Park et al., 2023,
    `claims.yaml`, `importance-poignancy-scale`) — deterministic, so the same event always
    scores the same. Small talk unrelated to any slot scores at the bottom of the scale; a
    restatement of an already-known fact scores lower than learning or changing it.
    """
    if slot is None or kind == "noise":
        return 1.0
    base = SLOTS[slot]["base_importance"]
    return base if kind in ("state_new", "update") else round(base * REPEAT_DISCOUNT, 2)


class Event(NamedTuple):
    """One turn of the interaction log."""

    t: int
    kind: str                    # "state_new" | "update" | "repeat" | "noise"
    slot: Optional[str]
    value: Optional[str]
    text: str
    importance: float


class Probe(NamedTuple):
    """One fact-recall question, asked at turn `t`, with the CURRENT correct answer at that turn."""

    t: int
    slot: str
    correct_value: str
    query_text: str


def generate_interaction_log(n_turns: int, seed: int) -> tuple[list[Event], list[Probe]]:
    """Given: build one long, seeded interaction log — deterministic for a fixed (n_turns, seed).

    Every slot is introduced once, early, on a fixed schedule. After a warm-up, each later turn
    is an UPDATE (new value for a known slot, ~12% of turns), a REPEAT (the known slot's
    CURRENT value, in a template different from the one last used for it, ~20% of turns), or
    NOISE (~68%, plus every turn before warm-up). A probe is recorded for every known slot every
    `PROBE_EVERY` turns after warm-up, holding that slot's value AS OF that turn.
    """
    rng = np.random.default_rng(seed)
    slot_names = list(SLOTS)
    n_slots = len(slot_names)
    span = max(1, n_turns // (3 * n_slots))
    introduce_at = {slot: 2 + i * span for i, slot in enumerate(slot_names)}
    warmup = max(introduce_at.values()) + 5

    current: dict[str, str] = {}
    last_template: dict[str, int] = {}
    events: list[Event] = []
    probes: list[Probe] = []

    for t in range(n_turns):
        introduced_now = [s for s in slot_names if introduce_at[s] == t]
        if introduced_now:
            slot = introduced_now[0]
            value = str(rng.choice(SLOTS[slot]["values"]))
            tmpl_i = int(rng.integers(0, len(SLOTS[slot]["templates"])))
            text = SLOTS[slot]["templates"][tmpl_i].format(v=value)
            current[slot] = value
            last_template[slot] = tmpl_i
            events.append(Event(t, "state_new", slot, value, text,
                                 importance_oracle(slot, "state_new")))
        else:
            known = [s for s in slot_names if s in current]
            if not known or t < warmup:
                kind = "noise"
            else:
                roll = rng.random()
                kind = "update" if roll < 0.12 else "repeat" if roll < 0.32 else "noise"
            if kind == "noise":
                text = str(rng.choice(NOISE_PHRASES))
                events.append(Event(t, "noise", None, None, text, importance_oracle(None, "noise")))
            elif kind == "update":
                slot = str(rng.choice(known))
                others = [v for v in SLOTS[slot]["values"] if v != current[slot]]
                value = str(rng.choice(others))
                # A different template from the one just used for the OLD value: this is what
                # keeps an update's text lexically distinct from its own predecessor, so a
                # near-duplicate detector has a real signal to work with rather than two
                # sentences that differ by one word.
                other_templates = [i for i in range(len(SLOTS[slot]["templates"]))
                                    if i != last_template.get(slot)]
                tmpl_i = int(rng.choice(other_templates)) if other_templates else 0
                text = SLOTS[slot]["templates"][tmpl_i].format(v=value)
                current[slot] = value
                last_template[slot] = tmpl_i
                events.append(Event(t, "update", slot, value, text,
                                     importance_oracle(slot, "update")))
            else:  # repeat
                slot = str(rng.choice(known))
                value = current[slot]
                other_templates = [i for i in range(len(SLOTS[slot]["templates"]))
                                    if i != last_template.get(slot)]
                tmpl_i = int(rng.choice(other_templates)) if other_templates else last_template.get(slot, 0)
                text = SLOTS[slot]["templates"][tmpl_i].format(v=value)
                events.append(Event(t, "repeat", slot, value, text,
                                     importance_oracle(slot, "repeat")))

        if t >= warmup and (t - warmup) % PROBE_EVERY == 0:
            for slot in slot_names:
                if slot in current:
                    probes.append(Probe(t, slot, current[slot],
                                         f"What is the user's {slot.replace('_', ' ')}?"))

    return events, probes


def _show_log_sample() -> None:
    events, probes = generate_interaction_log(N_TURNS, SEED)
    kinds = {}
    for ev in events:
        kinds[ev.kind] = kinds.get(ev.kind, 0) + 1
    print(f"{len(events)} events over {N_TURNS} turns: " +
          ", ".join(f"{k}={v}" for k, v in sorted(kinds.items())))
    print(f"{len(probes)} probes\n")
    for ev in events[:4] + [e for e in events if e.kind == "update"][:2]:
        print(f"  t={ev.t:>3} {ev.kind:<9} {ev.text}")
    print(f"  ...\n  sample probe: {probes[len(probes) // 2]}")


_try("interaction log (given)", _show_log_sample)

## 4. Exercise 1 — `write_policy_threshold()`

The simplest write policy that is not "store everything": store an event as its own memory
only if its importance clears a threshold. This is still naive in two ways later exercises
fix — no deduplication, no update-handling — so a restated fact becomes a second record, and
an updated fact becomes a second record that CONTRADICTS the first (both `current=True`).

<details><summary>💡 Hint 1 — what to think about</summary>

Which events have no slot at all (small talk), and should never become a memory regardless of
their importance? Is the gate "importance greater than threshold" or "importance at least
threshold" — check what happens to an event whose importance is EXACTLY the threshold.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Walk the events in order. Skip one with no slot, or with importance below the threshold —
skipping is "less than", so equal to the threshold is kept. For everything else, embed its
text and build a `MemoryRecord` with a fresh id (`len(store)` before you append is the ID of
the record you are about to add, so ids come out 0, 1, 2, ... in order), `current=True`.
</details>

In [ ]:
class MemoryRecord(NamedTuple):
    """One stored memory."""

    id: int
    slot: str
    value: str
    text: str
    t: int                # turn this record was last (re)stated or merged at
    importance: float      # 1-10
    current: bool           # False once a later update has superseded it
    embedding: np.ndarray


def write_policy_threshold(events: Sequence[Event], threshold: float,
                            vocab: Sequence[str], idf: np.ndarray) -> list[MemoryRecord]:
    """Store every event whose importance clears `threshold`, each as its own memory record.

    No deduplication and no update-handling (exercises 2-4 add those): a restated fact becomes
    a second record, and an updated fact becomes a second record that contradicts the first
    (both left `current=True`). An event with no slot (small talk) is never stored, regardless
    of its importance.

    Example:
        >>> ev = Event(0, "state_new", "role", "cook", "I work as a cook.", 7.5)
        >>> v = build_vocab([ev.text]); idf = build_idf([ev.text], v)
        >>> store = write_policy_threshold([ev], 5.0, v, idf)
        >>> len(store), store[0].value, store[0].current
        (1, 'cook', True)

    Returns:
        The list of stored `MemoryRecord`s, in the order their events occurred.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def write_policy_store_everything(events: Sequence[Event],
                                   vocab: Sequence[str], idf: np.ndarray) -> list[MemoryRecord]:
    """Given: the "store everything" baseline named in the brief — every slot-bearing event
    becomes a record. Built on `write_policy_threshold` with a threshold below any real score.
    """
    return write_policy_threshold(events, float("-inf"), vocab, idf)


def _check_write_policy_threshold() -> None:
    events = [
        Event(0, "state_new", "role", "cook", "I work as a cook.", 7.0),
        Event(1, "noise", None, None, "Nice weather.", 9.0),   # small talk, oddly high score
        Event(2, "state_new", "commute", "bike", "I commute by bike.", 3.5),
    ]
    vocab = build_vocab([e.text for e in events])
    idf = build_idf([e.text for e in events], vocab)
    store = write_policy_threshold(events, 5.0, vocab, idf)
    assert len(store) == 1, (
        f"threshold 5.0 should keep only the role event (importance 7.0 >= 5.0), and drop both "
        f"the small-talk event (no slot, whatever its importance) and the commute event "
        f"(3.5 < 5.0), got {len(store)} records — if you got 2, check you skip `slot is None`"
    )
    assert store[0].value == "cook" and store[0].current is True, (
        "the stored record should carry the event's own value and start out current=True"
    )
    boundary = write_policy_threshold(events, 3.5, vocab, idf)
    assert len(boundary) == 2, (
        f"an event whose importance EQUALS the threshold (commute, 3.5) must be kept — the "
        f"gate is >=, not > — got {len(boundary)} records at threshold=3.5"
    )
    assert (boundary[1].t, boundary[1].importance) == (2, 3.5), (
        f"each record must carry its OWN event's turn and importance (commute: t=2, 3.5), got "
        f"t={boundary[1].t}, importance={boundary[1].importance}"
    )
    one_step_up = write_policy_threshold(events, 3.51, vocab, idf)
    assert len(one_step_up) == 1, (
        f"one step ABOVE the commute event's importance (threshold 3.51 > 3.5) it must be "
        f"dropped again — got {len(one_step_up)} records; is the gate loosened by a margin?"
    )
    assert boundary[1].embedding.shape == (len(vocab),), (
        "each record's embedding must be a vector over the full vocabulary"
    )
    print("write_policy_threshold looks right")


_try("exercise 1", _check_write_policy_threshold)

## 5. Exercise 2 — `find_duplicate()`

A restatement in different words is not a new fact — "Home for me right now is Lisbon" after
"I live in Lisbon now" is the SAME memory, said again. Before a new record is created, check
whether an existing one already covers it.

<details><summary>💡 Hint 1 — what to think about</summary>

A duplicate is a restatement of the SAME fact — should this ever compare a `home_city`
candidate against a `manager` memory? Should a memory that has already been superseded
(`current=False`) still be able to "catch" a new duplicate?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Filter the store down to current records for the given slot. Compute the candidate's cosine
similarity to each; keep the best one. If the best similarity is at least `sim_threshold`,
return that record's id — equal to the threshold counts, the same way exercise 1's gate does.
Otherwise return `None`. Break a tie between two equally-similar records by the lower id.
</details>

In [ ]:
def find_duplicate(candidate_embedding: np.ndarray, store: Sequence[MemoryRecord],
                    slot: str, sim_threshold: float) -> Optional[int]:
    """Return the id of the most similar CURRENT memory already stored for `slot`, or None.

    Searches only records with `record.slot == slot and record.current` — a duplicate is a
    restatement of the SAME fact, never a coincidental resemblance between two different facts,
    and a memory that has already been superseded cannot "catch" a new one. The best cosine
    similarity found is compared against `sim_threshold` with `>=`: an exact tie with the
    threshold counts as a duplicate. Ties between two equally-similar records are broken by the
    lower id.

    Example:
        >>> find_duplicate(some_home_city_embedding, store, "home_city", 0.35)
        3          # or None, if nothing stored for "home_city" is similar enough

    Returns:
        The matching record's `id`, or `None` if no current record for `slot` is similar enough.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_find_duplicate() -> None:
    a = np.array([1.0, 0.0, 0.0])
    sixty = np.array([0.6, 0.8, 0.0])               # a 3-4-5 triangle: cosine(a, sixty) is 0.6 exactly
    far = np.array([0.0, 0.0, 1.0])                 # orthogonal to every vector below: cosine 0.0

    store = [
        MemoryRecord(0, "home_city", "old", "...", 0, 8.5, False, a),     # right slot, superseded
        MemoryRecord(1, "role", "cook", "...", 0, 7.0, True, a),          # perfect match, WRONG slot
        MemoryRecord(2, "home_city", "Lisbon", "...", 0, 8.5, True, sixty),
    ]
    got = find_duplicate(a, store, "home_city", 0.5)
    assert got == 2, (
        f"only record 2 is both a home_city record and still current — got {got}. Record 0 is "
        f"superseded and record 1 is for another slot, however perfectly their vectors match: "
        f"filter on `slot` AND `current` before comparing anything"
    )
    assert find_duplicate(far, store, "home_city", 0.5) is None, (
        "cosine 0.0 is well below sim_threshold=0.5 — there is no duplicate here"
    )
    assert find_duplicate(a, store, "role", 0.99) == 1, (
        "record 1 has slot='role' — a candidate for 'role' must be able to match it"
    )
    assert find_duplicate(a, store, "home_city", 0.6) == 2, (
        "cosine similarity EXACTLY 0.6 must count as a duplicate when sim_threshold=0.6 — the "
        "gate is >=, not >"
    )
    assert find_duplicate(a, store, "home_city", 0.601) is None, (
        "one step above the only current match's cosine (0.601 > 0.6) there is no duplicate — "
        "is the gate loosened by a margin?"
    )
    print("find_duplicate looks right")


_try("exercise 2", _check_find_duplicate)

## 6. Exercise 3 — `merge_or_append()`

Wire exercise 2 into the write path: a new `state_new` or `repeat` event either merges into
an existing memory or becomes a new one.

<details><summary>💡 Hint 1 — what to think about</summary>

What should change on the EXISTING record when it absorbs a duplicate — its `t`? its
`importance`? Should the number of records in the store ever go DOWN when you merge? Should
it go UP?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Embed the event's text. Call `find_duplicate` against the slot's current records. No match:
append a brand-new `MemoryRecord`, exactly as `write_policy_threshold` builds one. A match:
rebuild the store with that one record replaced — same id, `t` moved to the later of the two
turns, `importance` the larger of the two (a fact mentioned twice is at least as memorable as
either mention alone) — and every other record unchanged.
</details>

In [ ]:
def merge_or_append(store: Sequence[MemoryRecord], event: Event,
                     vocab: Sequence[str], idf: np.ndarray, sim_threshold: float) -> list[MemoryRecord]:
    """Add one non-`update` event to `store`, merging it into a near-duplicate if one exists.

    Embeds `event.text` and calls `find_duplicate` against `store`, restricted to
    `event.slot`. A duplicate is MERGED, not appended: the matching record's `t` becomes
    `max(old t, event.t)` and its `importance` becomes `max(old importance, event.importance)`
    — the number of records does not change. No duplicate: a new record is appended, built the
    same way `write_policy_threshold` builds one, with id `len(store)`.

    Example:
        >>> len(merge_or_append(one_record_store, repeat_of_it, vocab, idf, 0.35))
        1          # merged into the existing record, not appended as a second one

    Returns:
        The new store — same length if merged, one longer if appended.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_merge_or_append() -> None:
    first = Event(0, "state_new", "home_city", "Lisbon", "I live in Lisbon now.", 8.5)
    repeat = Event(50, "repeat", "home_city", "Lisbon", "Home for me right now is Lisbon.", 5.1)
    distinct = Event(60, "state_new", "role", "cook", "I work as a cook.", 7.0)
    texts = [first.text, repeat.text, distinct.text]
    vocab = build_vocab(texts)
    idf = build_idf(texts, vocab)

    store = write_policy_threshold([first], 0.0, vocab, idf)
    merged = merge_or_append(store, repeat, vocab, idf, 0.2)
    assert len(merged) == 1, (
        f"'Home for me right now is Lisbon' restates the SAME fact as 'I live in Lisbon now' — "
        f"it should MERGE (store stays length 1), got length {len(merged)}. Check "
        f"find_duplicate is actually being consulted before you append"
    )
    assert merged[0].t == 50 and merged[0].importance == 8.5, (
        f"a merge should move t to the LATER turn (50) and keep the LARGER importance (8.5, "
        f"not the repeat's discounted 5.1) — got t={merged[0].t}, importance={merged[0].importance}"
    )

    grown = merge_or_append(merged, distinct, vocab, idf, 0.2)
    assert len(grown) == 2, (
        f"a genuinely distinct fact (a different slot) must be APPENDED, not merged away — "
        f"got length {len(grown)}, expected 2"
    )
    assert grown[1].id == 1 and grown[0].id == 0, (
        "appending must not disturb the existing record's id, and the new record's id should "
        "be the store's previous length"
    )
    print("merge_or_append looks right")


_try("exercise 3", _check_merge_or_append)


def write_policy_dedup(events: Sequence[Event], threshold: float, sim_threshold: float,
                        vocab: Sequence[str], idf: np.ndarray) -> list[MemoryRecord]:
    """Given: threshold-gated writes, merging near-duplicates (exercises 1-3). `update` events
    are still appended naively here, exactly as `write_policy_threshold` would — see
    `write_policy_full` below for what exercise 4 changes.
    """
    store: list[MemoryRecord] = []
    for ev in events:
        if ev.slot is None or ev.importance < threshold:
            continue
        if ev.kind == "update":
            emb = embed(ev.text, vocab, idf)
            store = list(store) + [MemoryRecord(len(store), ev.slot, ev.value, ev.text, ev.t,
                                                  ev.importance, True, emb)]
        else:
            store = merge_or_append(store, ev, vocab, idf, sim_threshold)
    return store

## 7. Exercise 4 — `apply_update()`

The trap this exercise exists to close: the user moves city, or gets a new manager. The OLD
value must stop being retrievable as the answer — not merely lose a close contest on points.

<details><summary>💡 Hint 1 — what to think about</summary>

How many CURRENT records for this slot might already be in the store — always exactly one?
What happens to fact recall later if you leave even one stale `current=True` record behind
for this slot?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Rebuild the store: every record whose slot matches the event's slot AND is still current gets
replaced by itself with `current=False` (there may be more than one, if an earlier write
policy skipped deduplication — supersede all of them, not just the first). Every other record
passes through unchanged. Then append a brand-new record for the event's new value, with
`current=True` and id `len(store)`.
</details>

In [ ]:
def apply_update(store: Sequence[MemoryRecord], event: Event,
                  vocab: Sequence[str], idf: np.ndarray) -> list[MemoryRecord]:
    """Apply one `kind == "update"` event: supersede the slot's current record(s), add the new one.

    Every record with `record.slot == event.slot and record.current` is replaced by itself with
    `current=False` (there is normally exactly one, but a store built without deduplication may
    hold several — supersede every one of them). A new `MemoryRecord` for `event.value` is then
    appended with `current=True` and id `len(store)`. Every record for a different slot is
    returned unchanged, in its original position.

    Example:
        >>> [(r.value, r.current) for r in apply_update(store_with_one_lisbon_record, moved_to_berlin, vocab, idf)]
        [('Lisbon', False), ('Berlin', True)]

    Returns:
        The new store, one record longer than `store`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def write_policy_full(events: Sequence[Event], threshold: float, sim_threshold: float,
                       vocab: Sequence[str], idf: np.ndarray) -> list[MemoryRecord]:
    """Given: the fully-managed policy — threshold-gated (1), deduplicated (2, 3) AND
    supersession-aware (4). Composes every graded exercise in this lesson.
    """
    store: list[MemoryRecord] = []
    for ev in events:
        if ev.slot is None or ev.importance < threshold:
            continue
        if ev.kind == "update":
            store = apply_update(store, ev, vocab, idf)
        else:
            store = merge_or_append(store, ev, vocab, idf, sim_threshold)
    return store


def write_policy_supersede_only(events: Sequence[Event], threshold: float,
                                 vocab: Sequence[str], idf: np.ndarray) -> list[MemoryRecord]:
    """Given: threshold-gated and supersession-aware, but NOT deduplicated — every restatement
    of an already-current value is appended as its own new, current record, so a slot can hold
    several current records until its next update supersedes them all (exercise 4's job).
    The trap-2 demo below compares this against `write_policy_full`, holding supersession ON
    in both, so the only difference between the two is deduplication. With supersession OFF in
    both, a repeat could merge into a stale, wrong-value record in one arm and not the other.
    """
    store: list[MemoryRecord] = []
    for ev in events:
        if ev.slot is None or ev.importance < threshold:
            continue
        if ev.kind == "update":
            store = apply_update(store, ev, vocab, idf)
        else:
            emb = embed(ev.text, vocab, idf)
            store = list(store) + [MemoryRecord(len(store), ev.slot, ev.value, ev.text, ev.t,
                                                  ev.importance, True, emb)]
    return store


def _check_apply_update() -> None:
    lisbon = Event(0, "state_new", "home_city", "Lisbon", "I live in Lisbon now.", 8.5)
    berlin = Event(50, "update", "home_city", "Berlin", "I live in Berlin now.", 8.5)
    role = Event(10, "state_new", "role", "cook", "I work as a cook.", 7.0)
    texts = [lisbon.text, berlin.text, role.text]
    vocab = build_vocab(texts)
    idf = build_idf(texts, vocab)

    store = write_policy_threshold([lisbon, role], 0.0, vocab, idf)
    updated = apply_update(store, berlin, vocab, idf)
    assert len(updated) == 3, f"an update should ADD a record, not replace one in place — got length {len(updated)}, expected 3"
    lisbon_rec = next(r for r in updated if r.value == "Lisbon")
    berlin_rec = next(r for r in updated if r.value == "Berlin")
    role_rec = next(r for r in updated if r.value == "cook")
    assert lisbon_rec.current is False, (
        "the OLD home_city record (Lisbon) must be marked current=False once it has been "
        "superseded, or it will still win a future retrieval and give the wrong answer"
    )
    assert berlin_rec.current is True and berlin_rec.t == 50, (
        "the NEW home_city record (Berlin) must be current=True at the update's own turn"
    )
    assert role_rec.current is True, (
        "a record for a DIFFERENT slot must be left exactly as it was — apply_update must not "
        "touch every current record, only the ones for event.slot"
    )

    double = write_policy_threshold([lisbon], 0.0, vocab, idf)
    double = double + [double[0]._replace(id=1, t=5)]     # simulate a second stale 'current' record
    fixed = apply_update(double, berlin, vocab, idf)
    stale_survivors = [r for r in fixed if r.value == "Lisbon" and r.current]
    assert stale_survivors == [], (
        "if MORE than one current record exists for a slot (e.g. deduplication was skipped "
        "upstream), apply_update must supersede ALL of them, not just the first one found"
    )
    print("apply_update looks right")


_try("exercise 4", _check_apply_update)

## 8. Exercise 5 — `recency_score()`

Park et al. (2023) score how recent a memory is with an exponential decay over elapsed time
(`claims.yaml`, `recency-exponential-decay`). Get the DIRECTION right: which way does a
larger decay constant push the score? One honest difference: the paper counts game hours
since a memory was last RETRIEVED; this lesson counts turns since it was last STATED, so
reading the store never changes it.

<details><summary>💡 Hint 1 — what to think about</summary>

At `elapsed_turns = 0` — a memory stated this very turn — what should the recency score be,
regardless of `decay`? Does raising a number between 0 and 1 to a HIGHER power make it bigger
or smaller?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

One line: the decay constant raised to the power of the elapsed turns. Nothing about "how
many turns are left" or "how close to now" — just the constant, to that power.
</details>

In [ ]:
def recency_score(elapsed_turns: int, decay: float) -> float:
    """Park et al. (2023)'s recency term: exponential decay over elapsed time.

    `decay` is a constant in (0, 1); the paper's own value is 0.995 per simulated hour. This
    lesson counts in interaction TURNS rather than hours, but the formula — and which way it
    points — is exactly theirs: `recency = decay ** elapsed_turns`. Zero turns elapsed gives
    1.0 (perfectly fresh, for any decay). The LARGER `decay` is (closer to 1), the SLOWER a
    memory fades: raising a number nearer 1 to a power shrinks it more slowly than raising a
    number nearer 0 to the same power.

    Example:
        >>> round(recency_score(0, 0.9), 6)
        1.0
        >>> round(recency_score(1, 0.9), 6)
        0.9

    Returns:
        The recency score, in (0, 1] for `elapsed_turns >= 0`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_recency_score() -> None:
    assert abs(recency_score(0, 0.99) - 1.0) < 1e-12, (
        "elapsed_turns=0 must give exactly 1.0 for ANY decay — a memory stated this very turn "
        "has not faded at all yet"
    )
    assert abs(recency_score(1, 0.9) - 0.9) < 1e-9, (
        f"recency_score(1, 0.9) should be exactly the decay constant itself (0.9**1), got "
        f"{recency_score(1, 0.9)}"
    )
    slow = recency_score(50, 0.99)
    fast = recency_score(50, 0.90)
    assert slow > fast, (
        f"at the SAME elapsed time, the LARGER decay constant (0.99) must give the HIGHER "
        f"recency score (slower fade), got recency(0.99)={slow:.4f} <= recency(0.90)={fast:.4f} "
        f"— this is the decay constant's direction; a swapped exponent or a (1 - decay) term "
        f"gets it backwards"
    )
    later = recency_score(51, 0.99)
    assert later < slow, (
        f"recency must strictly DECREASE as elapsed_turns grows, for a fixed decay < 1 — got "
        f"recency(50)={slow:.6f} <= recency(51)={later:.6f}"
    )
    print("recency_score looks right")


_try("exercise 5", _check_recency_score)

## 9. Exercise 6 — `retrieve_top_k()`

Combine recency, importance and relevance into the one score Park et al. (2023) retrieve by
(`claims.yaml`, `retrieval-score-combination`): each normalised to `[0, 1]` over the pool
being scored, then summed with equal weight. This is also where the memory budget lives —
`budget_k` is the most memories a retrieval is ever allowed to spend.

<details><summary>💡 Hint 1 — what to think about</summary>

Should a superseded record ever be a candidate? What does "normalise over the pool" mean when
the pool has only one candidate, or when every candidate has the same importance?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Filter the store to `current` records — that alone is what makes an update's old value stop
being retrievable. Build three arrays over the surviving candidates: recency (exercise 5,
elapsed = `now_t - record.t`), importance (the record's own field) and relevance (cosine
between `query_embedding` and each record's embedding). Min-max scale each array to `[0, 1]`
(given, as `_minmax`) and add the three scaled arrays. Sort by score descending, id ascending
to break a tie, and keep the first `budget_k`.
</details>

In [ ]:
def _minmax(values: np.ndarray) -> np.ndarray:
    """Given: min-max scale `values` to [0, 1]. A constant array (nothing to tell apart) maps
    to all 1.0, rather than dividing by zero — nobody is penalised for a three-way tie."""
    lo, hi = float(values.min()), float(values.max())
    if hi - lo < 1e-12:
        return np.ones_like(values)
    return (values - lo) / (hi - lo)


def retrieve_top_k(query_embedding: np.ndarray, store: Sequence[MemoryRecord], now_t: int,
                    budget_k: int, decay: float) -> list[MemoryRecord]:
    """Score every CURRENT memory in `store` and return its top `budget_k` by combined score.

    Follows Park et al. (2023)'s retrieval function: recency (`recency_score`, exercise 5, with
    `elapsed_turns = now_t - record.t`), importance (the record's own field) and relevance
    (`cosine(query_embedding, record.embedding)`) are each min-max normalised to `[0, 1]` over
    the candidates being scored (`_minmax`, given), then summed with equal weight — `score =
    recency' + importance' + relevance'`, exactly as the paper's own implementation (all three
    weights equal 1). A superseded record (`current=False`) is never a candidate: it is not
    scored, and cannot be retrieved. Ties in score are broken by the lower id. An empty
    candidate pool returns `[]`.

    Example:
        >>> len(retrieve_top_k(some_query_embedding, store, now_t=100, budget_k=2, decay=0.99))
        2   # or fewer, if fewer than 2 current records exist

    Returns:
        At most `budget_k` `MemoryRecord`s, most relevant first.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_retrieve_top_k() -> None:
    q = np.array([1.0, 0.0])
    near = np.array([1.0, 0.0])     # cosine 1.0 with q
    far = np.array([0.0, 1.0])       # cosine 0.0 with q

    empty = retrieve_top_k(q, [], now_t=10, budget_k=3, decay=0.9)
    assert empty == [], "an empty store must retrieve an empty list, not raise"

    # (A) superseded records must never be returned, however good their other scores are.
    superseded = MemoryRecord(0, "home_city", "old", "...", 10, 10.0, False, near)
    live = MemoryRecord(1, "role", "cook", "...", 10, 1.0, True, far)
    out = retrieve_top_k(q, [superseded, live], now_t=10, budget_k=2, decay=0.9)
    assert [r.id for r in out] == [1], (
        f"a superseded record (current=False) must be excluded from retrieval entirely, "
        f"however high its recency/importance/relevance — got ids {[r.id for r in out]}"
    )

    # (B) importance must be able to overcome a recency gap — a "recency-only" mistake fails
    # this. The third, oldest and off-topic memory keeps min-max scaling from turning the
    # contest between the other two into an exact tie.
    recent_trivial = MemoryRecord(0, "coffee_order", "v", "...", 199, 1.0, True, near)
    old_important = MemoryRecord(1, "home_city", "v", "...", 100, 10.0, True, near)
    oldest_off_topic = MemoryRecord(2, "commute", "v", "...", 0, 5.0, True, far)
    top1 = retrieve_top_k(q, [recent_trivial, old_important, oldest_off_topic], now_t=200,
                          budget_k=1, decay=0.99)
    assert top1[0].id == 1, (
        f"with equal relevance, the OLDER but IMPORTANT memory (id 1, importance 10) should "
        f"beat the RECENT but trivial one (id 0, importance 1) once importance is genuinely "
        f"added into the score — got top-1 id={top1[0].id}. A retrieval that scores by recency "
        f"alone gets this backwards"
    )

    # (C) relevance must matter too — two equally recent, equally important records, one on-topic.
    a = MemoryRecord(0, "home_city", "v", "...", 50, 5.0, True, far)
    b = MemoryRecord(1, "home_city", "v", "...", 50, 5.0, True, near)
    top1b = retrieve_top_k(q, [a, b], now_t=50, budget_k=1, decay=0.99)
    assert top1b[0].id == 1, (
        "with equal recency and importance, the record whose embedding is actually similar to "
        "the query must be preferred — check relevance is really in the sum"
    )

    # (D) the budget caps how many come back.
    many = [MemoryRecord(i, "home_city", "v", "...", 50, 5.0, True, near) for i in range(5)]
    capped = retrieve_top_k(q, many, now_t=50, budget_k=2, decay=0.99)
    assert len(capped) == 2, f"budget_k=2 must return at most 2 records, got {len(capped)}"

    # (E) recency must count, and point the right way: equal importance, equal relevance.
    by_age = [MemoryRecord(i, "home_city", "v", "...", t, 5.0, True, near)
              for i, t in enumerate((10, 50, 90))]
    ranked = [r.id for r in retrieve_top_k(q, by_age, now_t=100, budget_k=3, decay=0.99)]
    assert ranked == [2, 1, 0], (
        f"with equal importance and relevance, the most recently stated memory must rank first "
        f"— got ids {ranked}, expected [2, 1, 0]. Check recency is in the sum, with "
        f"elapsed_turns = now_t - record.t (not the other way round)"
    )

    # (F) each signal is min-max scaled over the CURRENT candidates before the three are added.
    # Importance spans only 5.0-5.5 here and relevance 0.0-0.8; scaled, both span [0, 1].
    pool = [
        MemoryRecord(0, "role", "v", "...", 50, 5.0, True, np.array([0.8, 0.6])),   # relevance 0.8
        MemoryRecord(1, "role", "v", "...", 50, 5.0, True, far),                    # relevance 0.0
        MemoryRecord(2, "role", "v", "...", 50, 5.5, True, np.array([0.2, np.sqrt(0.96)])),
        MemoryRecord(3, "role", "v", "...", 50, 10.0, False, far),                  # superseded
    ]
    top = retrieve_top_k(q, pool, now_t=50, budget_k=1, decay=0.99)[0].id
    assert top == 2, (
        f"expected id 2 (top importance, some relevance) on top, got id {top}: raw, unscaled "
        f"sums, or min-max scaling that also counts the superseded record, both pick id 0 — "
        f"scale each signal with _minmax over the current candidates only, then add them"
    )
    print("retrieve_top_k looks right")


_try("exercise 6", _check_retrieve_top_k)

## 10. Measuring what matters: fact recall against memory size and retrieval cost

The metric this whole lesson has been building towards: for a probe question asked at a
later turn, does retrieval return the memory holding the CURRENT correct value? Measured
across policies and a swept budget, this is what turns "which write policy?" from a taste
into a number.

One honest limit shows in the first lines it prints: at this lesson's `SIM_THRESHOLD`, a few
restatements score below the gate and stay separate current records, so even the full policy
holds more current records than there are facts. A word-count embedding cannot perfectly tell
"the same fact, reworded" from "the same slot, a new value".

In [ ]:
def fact_recall(build_store: Callable[[Sequence[Event]], list[MemoryRecord]],
                 events: Sequence[Event], probes: Sequence[Probe],
                 vocab: Sequence[str], idf: np.ndarray,
                 budget_k: int, decay: float) -> tuple[float, float]:
    """Given: fact recall and mean retrieval cost for one write policy, swept over `probes`.

    For each probe, the store is rebuilt from SCRATCH using only the events up to and including
    the probe's own turn — `build_store(events[:probe.t + 1])`. This matters, not just filters
    the final store by `t <= probe.t`: `current` can flip to False at a turn LATER than a
    record's own `t` (that is what supersession IS), so filtering the finished store by `t`
    alone would let a supersession that has not happened yet, as of the probe, leak backwards
    in time. Rebuilding from the truncated log is what a real memory store would actually have
    held at that moment. `build_store` is one of this lesson's write policies with its own
    threshold/sim_threshold already bound (see the demo cells below). A probe counts as a hit
    if ANY retrieved record names `probe.slot` with `probe.correct_value`.
    """
    hits = 0
    total_scored = 0
    for pr in probes:
        as_of = build_store(events[: pr.t + 1])
        total_scored += sum(1 for rec in as_of if rec.current)
        q = embed(pr.query_text, vocab, idf)
        top = retrieve_top_k(q, as_of, pr.t, budget_k, decay)
        if any(rec.slot == pr.slot and rec.value == pr.correct_value for rec in top):
            hits += 1
    n = len(probes)
    return (hits / n if n else 0.0), (total_scored / n if n else 0.0)


def _policies(vocab: Sequence[str], idf: np.ndarray) -> dict[str, Callable[[Sequence[Event]], list[MemoryRecord]]]:
    """Given: the four write policies compared throughout this lesson, as `events -> store`
    callables with their own parameters bound, ready for `fact_recall`."""
    return {
        "store everything": lambda evs: write_policy_store_everything(evs, vocab, idf),
        "threshold only": lambda evs: write_policy_threshold(evs, THRESHOLD, vocab, idf),
        "threshold + dedup": lambda evs: write_policy_dedup(evs, THRESHOLD, SIM_THRESHOLD, vocab, idf),
        "threshold + supersede": lambda evs: write_policy_supersede_only(evs, THRESHOLD, vocab, idf),
        "threshold + dedup + supersede (full)":
            lambda evs: write_policy_full(evs, THRESHOLD, SIM_THRESHOLD, vocab, idf),
    }


def _show_recall_cost_curve() -> None:
    events, probes = generate_interaction_log(N_TURNS, SEED)
    vocab = build_vocab([e.text for e in events] + [p.query_text for p in probes])
    idf = build_idf([e.text for e in events], vocab)
    policies = _policies(vocab, idf)

    for name, build_store in policies.items():
        store = build_store(events)
        print(f"{name:<38} {len(store):>3} records stored "
              f"({sum(r.current for r in store)} current)")

    budgets = (2, 4, 6, 10, 16, 24)
    print(f"\n{'policy':<38}" + "".join(f"k={k:<6}" for k in budgets))
    results: dict[str, list[tuple[float, float]]] = {}
    for name, build_store in policies.items():
        row = [fact_recall(build_store, events, probes, vocab, idf, k, DECAY) for k in budgets]
        results[name] = row
        print(f"{name:<38}" + "".join(f"{r:<7.2f}" for r, _ in row))

    best_name, best_k, best_recall, best_cost = None, None, -1.0, None
    for name, row in results.items():
        for k, (recall, cost) in zip(budgets, row):
            if recall > best_recall or (recall == best_recall and (best_cost is None or cost < best_cost)):
                best_name, best_k, best_recall, best_cost = name, k, recall, cost
    print(f"\nchosen policy: '{best_name}' at budget_k={best_k} "
          f"— recall {best_recall:.2f}, mean {best_cost:.0f} candidates scored per question")


_try("recall/cost curve", _show_recall_cost_curve,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 5", "exercise 6"))

## 11. The three traps, measured

Each of these prints a comparison rather than asserting one — read the printed numbers, not
this paragraph, for what actually happened on this machine's run.

In [ ]:
def _show_supersession_trap() -> None:
    events, probes = generate_interaction_log(N_TURNS, SEED)
    vocab = build_vocab([e.text for e in events] + [p.query_text for p in probes])
    idf = build_idf([e.text for e in events], vocab)
    first_update: dict[str, int] = {}
    for ev in events:
        if ev.kind == "update":
            first_update.setdefault(ev.slot, ev.t)
    # only a probe asked AFTER its fact first changed has a stale old value to be fooled by
    affected = [p for p in probes if p.slot in first_update and first_update[p.slot] <= p.t]
    if not affected:
        print("no probe was asked after its fact had changed in this run — nothing to show")
        return

    policies = _policies(vocab, idf)
    k = 3
    recall_stale, _ = fact_recall(policies["threshold + dedup"], events, affected, vocab, idf,
                                   budget_k=k, decay=DECAY)
    recall_current, _ = fact_recall(policies["threshold + dedup + supersede (full)"], events,
                                     affected, vocab, idf, budget_k=k, decay=DECAY)
    print(f"probes asked after their fact had changed at least once: {len(affected)} "
          f"of {len(probes)}, at k={k}")
    print(f"  recall WITHOUT supersession (exercise 4 skipped): {recall_stale:.2f}")
    print(f"  recall WITH supersession (write_policy_full):      {recall_current:.2f}")
    if recall_current > recall_stale:
        print("  a stale, still-'current' old value keeps competing with the new one on "
              "points; supersession removes it from the contest entirely.")
    else:
        print("  supersession did not raise recall on this log — look at which probes it lost.")


_try("trap 1 — supersession", _show_supersession_trap, needs=("exercise 4", "exercise 6"))


def _show_dedup_trap() -> None:
    """Compares 'supersede only' against 'full' (dedup ADDED on top), holding supersession
    fixed at ON in both — see `write_policy_supersede_only`'s docstring for why: with
    supersession on, every stale value is already current=False, so a repeat can only merge
    into a record holding the slot's current value, and the comparison isolates deduplication.
    """
    events, probes = generate_interaction_log(N_TURNS, SEED)
    vocab = build_vocab([e.text for e in events] + [p.query_text for p in probes])
    idf = build_idf([e.text for e in events], vocab)
    policies = _policies(vocab, idf)
    small_k = 4
    recall_a, cost_a = fact_recall(policies["threshold + supersede"], events, probes, vocab, idf,
                                    small_k, DECAY)
    recall_b, cost_b = fact_recall(policies["threshold + dedup + supersede (full)"], events, probes,
                                    vocab, idf, small_k, DECAY)
    print(f"at a small budget (k={small_k}):")
    print(f"  recall, supersede only (duplicates crowd the budget): {recall_a:.2f}, "
          f"mean {cost_a:.0f} candidates scored per question")
    print(f"  recall, dedup + supersede (merged, not crowding):      {recall_b:.2f}, "
          f"mean {cost_b:.0f} candidates scored per question")


_try("trap 2 — near-duplicate crowding", _show_dedup_trap, needs=("exercise 3", "exercise 6"))


def _show_pure_recency_trap() -> None:
    old_text = "My home city these days is Somewhere."
    corpus = [old_text, "The office coffee machine is broken again.",
              "What is the user's home city?"]
    vocab = build_vocab(corpus)
    idf = build_idf(corpus, vocab)
    q = embed("What is the user's home city?", vocab, idf)
    old_important = MemoryRecord(0, "home_city", "Somewhere", old_text,
                                  0, 8.5, True, embed(old_text, vocab, idf))
    recent_noise = [
        MemoryRecord(i + 1, "noise", "n", "The office coffee machine is broken again.",
                     195 + i, 1.0, True, embed("The office coffee machine is broken again.", vocab, idf))
        for i in range(6)
    ]
    pool = [old_important] + recent_noise
    now_t = 200

    def recency_only_top1(pool, now_t):
        scored = [(recency_score(now_t - r.t, DECAY), r) for r in pool]
        return max(scored, key=lambda pair: pair[0])[1]

    combined = retrieve_top_k(q, pool, now_t, budget_k=1, decay=DECAY)[0]
    by_recency = recency_only_top1(pool, now_t)
    ages = [now_t - r.t for r in recent_noise]
    print(f"pool: 1 memory {now_t - old_important.t} turns old with importance "
          f"{old_important.importance}, plus {len(recent_noise)} recent small-talk memories "
          f"(importance {recent_noise[0].importance}, {min(ages)}-{max(ages)} turns old)")
    print(f"  top-1 by recency ALONE:                 '{by_recency.text}' (importance {by_recency.importance})")
    print(f"  top-1 by the combined score (exercise 6): '{combined.text}' (importance {combined.importance})")
    if combined.id == old_important.id and by_recency.id != old_important.id:
        print("  pure recency forgets the old-but-important fact; the combined score keeps it.")


_try("trap 3 — pure recency", _show_pure_recency_trap, needs=("exercise 5", "exercise 6"))

## 12. Common mistakes

- Computing `recency_score` as `1 - decay ** elapsed` or `decay ** -elapsed` — either one
  makes OLDER memories score higher, which is backwards; check exercise 5's direction test.
- Restricting `find_duplicate` to the whole store instead of one slot — two different facts
  that happen to share a few words then "merge" into nonsense.
- Filtering `current=False` records everywhere EXCEPT inside `retrieve_top_k` itself, so a
  superseded memory still gets scored (and sometimes wins) even though `apply_update` marked
  it correctly.
- Min-max normalising recency, importance and relevance ACROSS PROBES or across the whole
  store instead of over the candidates actually being scored for one query — the ranking this
  produces is not what Park et al. (2023) describe.

## 13. Self-check

1. A memory was stored 200 turns ago with importance 9, and has never been mentioned since.
   Ten memories from the last five turns, each with importance 2, are otherwise unrelated to
   the current question. Under the combined retrieval score, is the old memory guaranteed to
   be excluded from a budget of 1?
   - (a) No — a high enough importance can still win, depending on `decay` and the ten
     memories' own relevance to the question
   - (b) Yes, because superseded memories are always dropped first
   - (c) Yes, importance alone can never outweigh 200 turns of recency decay

2. `recency_score(elapsed, decay)` uses `decay ** elapsed`. If you doubled `decay` from 0.9 to
   1.8, what goes wrong first?
   - (a) Nothing — decay is just a number, any value works
   - (b) The function raises a `ZeroDivisionError`
   - (c) `recency_score` can exceed 1.0 and grow with elapsed time instead of shrinking,
     because a base greater than 1 raised to a growing power increases

3. A store has two records for `home_city`: one `current=True` (the new value) and one
   `current=False` (the old value, which `apply_update` has correctly superseded). What
   does `retrieve_top_k` do with the superseded one?
   - (a) Scores it like any other candidate; it may still be returned
   - (b) Never scores it or returns it, because it is filtered out before scoring begins
   - (c) Automatically deletes it from the store

Answers come with this lesson's worked solution when you enrol on Synapsa.

## What you built, and where it goes next

A memory store that resists both traps a naive one falls into — duplicates crowding the
budget, a corrected fact losing to its own stale self — scored by recency, importance and
relevance the way Park et al. (2023) do, measured against FACT RECALL against memory size
and cost. Track T07's agent loop is where this store gets called from; T06-L01 is the search
machinery `embed` and `cosine` are a smaller cousin of.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


if __name__ == "__main__":
    # Every check is re-run here, quietly, against your code as it stands now — each one
    # already printed its feedback in its own cell above — so the board is current even if you
    # edited an exercise and did not re-run its cell.
    _ALL_CHECKS = (
        ("exercise 1", _check_write_policy_threshold),
        ("exercise 2", _check_find_duplicate),
        ("exercise 3", _check_merge_or_append),
        ("exercise 4", _check_apply_update),
        ("exercise 5", _check_recency_score),
        ("exercise 6", _check_retrieve_top_k),
    )
    import contextlib
    import io
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in _ALL_CHECKS:
            _try(_name, _check)
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.